In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import datetime, pandas

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Union
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
from abc import ABC, abstractmethod

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [2]:
train = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv').dropna()
test = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/test.csv').dropna()

In [3]:
train.columns

Index(['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1',
       'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
       'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'I1', 'I2', 'I3',
       'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M1', 'M10', 'M11', 'M12', 'M13',
       'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7',
       'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5',
       'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4',
       'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V10', 'V11', 'V12', 'V13', 'V2',
       'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'forward_returns',
       'risk_free_rate', 'market_forward_excess_returns'],
      dtype='object')

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
import warnings

def preprocessing(data, typ, 
                 imputation_strategy='knn', 
                 scaling_method='robust',
                 handle_outliers=True,
                 outlier_threshold=3,
                 feature_engineering=True,
                 verbose=True):
    
    # Define main features with proper organization
    feature_groups = {
        'market_dynamics': ['M' + str(i) for i in range(1, 19)],
        'economic': ['E' + str(i) for i in range(1, 21)],
        'interest_rate': ['I' + str(i) for i in range(1, 10)],
        'price_valuation': ['P8', 'P9', 'P10', 'P12', 'P13'],
        'volatility': ['V' + str(i) for i in range(1, 14)],
        'sentiment': ['S' + str(i) for i in range(1, 13)],
        'dummy': ['D' + str(i) for i in range(1, 10)]
    }
    
    # Flatten all features
    main_features = []
    for group_features in feature_groups.values():
        main_features.extend(group_features)
    
    # Select relevant columns
    if typ == "train":
        target_col = "market_forward_excess_returns"
        if target_col in data.columns:
            selected_cols = main_features + [target_col]
        else:
            warnings.warn(f"Target column '{target_col}' not found in training data")
            selected_cols = main_features
    else:
        selected_cols = main_features
    
    # Filter for existing columns only
    available_cols = [col for col in selected_cols if col in data.columns]
    missing_cols = set(selected_cols) - set(available_cols)
    
    if missing_cols and verbose:
        print(f"Warning: Missing columns: {missing_cols}")
    
    data_processed = data[available_cols].copy()
    
    if verbose:
        print(f"Processing {len(data_processed)} rows with {len(available_cols)} features")
        print(f"Missing values before processing: {data_processed.isnull().sum().sum()}")
    
    # Separate features and target
    if typ == "train" and target_col in data_processed.columns:
        features = [col for col in data_processed.columns if col != target_col]
        X = data_processed[features]
        y = data_processed[target_col]
    else:
        features = list(data_processed.columns)
        X = data_processed[features]
        y = None
    
    # Store metadata
    metadata = {'feature_groups': feature_groups}
    
    # Handle missing values
    if imputation_strategy == 'zero':
        X = X.fillna(0)
    elif imputation_strategy == 'median':
        X = X.fillna(X.median())
    elif imputation_strategy == 'forward_fill':
        X = X.fillna(method='ffill').fillna(0)  # Forward fill then zero for remaining
    elif imputation_strategy == 'knn':
        # Use KNN imputation for more sophisticated missing value handling
        imputer = KNNImputer(n_neighbors=5)
        feature_names = X.columns
        X_imputed = imputer.fit_transform(X)
        X = pd.DataFrame(X_imputed, columns=feature_names, index=X.index)
        metadata['imputer'] = imputer
    
    # Handle outliers using IQR method
    if handle_outliers:
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        outlier_info = {}
        
        for col in numeric_cols:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - outlier_threshold * IQR
            upper_bound = Q3 + outlier_threshold * IQR
            
            # Count outliers
            outliers = ((X[col] < lower_bound) | (X[col] > upper_bound)).sum()
            if outliers > 0:
                outlier_info[col] = outliers
                # Cap outliers instead of removing them
                X[col] = X[col].clip(lower=lower_bound, upper=upper_bound)
        
        if verbose and outlier_info:
            print(f"Capped outliers in {len(outlier_info)} columns: {dict(list(outlier_info.items())[:5])}")
        
        metadata['outlier_info'] = outlier_info
    
    # Feature Engineering
    if feature_engineering:
        # Create rolling statistics for time-sensitive features
        if 'date_id' in data.columns:
            # Sort by date for proper time series operations
            data_with_date = data.set_index('date_id').sort_index()
            
            # Rolling features for key metrics (example with market dynamics)
            market_cols = [col for col in X.columns if col.startswith('M')]
            for col in market_cols[:5]:  # Limit to avoid too many features
                if col in X.columns:
                    X[f'{col}_ma5'] = data_with_date[col].rolling(5).mean()
                    X[f'{col}_std5'] = data_with_date[col].rolling(5).std()
        
        # Create interaction features between different groups
        # Example: Volatility * Economic indicators
        vol_cols = [col for col in X.columns if col.startswith('V')][:3]
        econ_cols = [col for col in X.columns if col.startswith('E')][:3]
        
        for v_col in vol_cols:
            for e_col in econ_cols:
                if v_col in X.columns and e_col in X.columns:
                    X[f'{v_col}_{e_col}_interaction'] = X[v_col] * X[e_col]
        
        if verbose:
            print(f"Added feature engineering - new shape: {X.shape}")
    
    # Feature Scaling
    scaler = None
    if scaling_method == 'standard':
        scaler = StandardScaler()
    elif scaling_method == 'robust':
        scaler = RobustScaler()
    elif scaling_method == 'minmax':
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
    
    if scaler is not None:
        # Only scale numeric columns
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        X_scaled = X.copy()
        X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])
        X = X_scaled
        metadata['scaler'] = scaler
    
    # Combine features and target back
    if y is not None:
        result = pd.concat([X, y], axis=1)
    else:
        result = X
    
    # Final data quality check
    if verbose:
        print(f"Final shape: {result.shape}")
        print(f"Missing values after processing: {result.isnull().sum().sum()}")
        print(f"Data types: {result.dtypes.value_counts().to_dict()}")
        
    for i in zip(result.columns, result.dtypes):
        result[i[0]].fillna(0, inplace=True)
        
    return result, metadata

processed_data_train, metadata_train = preprocessing(train, "train")
processed_data_test, metadata_test = preprocessing(test, "test")

Processing 2021 rows with 87 features
Missing values before processing: 0
Capped outliers in 35 columns: {'M1': 1, 'M3': 47, 'M4': 29, 'M6': 59, 'M9': 5}
Added feature engineering - new shape: (2021, 105)
Final shape: (2021, 106)
Missing values after processing: 40
Data types: {dtype('float64'): 106}
Processing 10 rows with 86 features
Missing values before processing: 0
Capped outliers in 14 columns: {'M4': 1, 'E7': 2, 'E8': 1, 'E16': 2, 'E17': 2}
Added feature engineering - new shape: (10, 105)
Final shape: (10, 105)
Missing values after processing: 100
Data types: {dtype('float64'): 105}


In [5]:
processed_data_train

,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,...,V1_E1_interaction,V1_E2_interaction,V1_E3_interaction,V2_E1_interaction,V2_E2_interaction,V2_E3_interaction,V3_E1_interaction,V3_E2_interaction,V3_E3_interaction,market_forward_excess_returns
6969,-0.525806,-0.024123,-0.275782,0.048319,-2.325944,0.849620,-0.259874,-0.004827,0.652894,1.639214,...,-0.039664,0.945315,0.110245,-0.057286,1.031166,0.094016,0.020596,1.109477,0.122149,0.000797
6970,-0.517250,-0.061453,-0.012835,-0.255548,-2.105498,0.895303,-0.259169,0.717619,0.656737,1.643486,...,-0.026713,0.969908,0.125261,-0.058761,1.037199,0.102580,0.028330,1.131193,0.136451,0.004390
6971,-0.577286,-0.452395,-0.495483,0.020945,-2.206294,0.941953,-0.258465,0.545455,0.681975,1.669823,...,-0.036566,1.001977,0.184221,-0.060234,1.083810,0.165858,0.005168,1.141689,0.190958,0.005669
6972,-0.654268,-0.852348,-0.623131,0.607230,-2.321339,0.502814,-0.257761,-1.186645,0.712855,1.677442,...,-0.054656,1.032458,0.253520,-0.061707,1.142206,0.245661,-0.011279,1.173722,0.264080,0.001067
6973,-0.756118,-0.738989,-0.494108,0.656315,-2.275654,0.246790,-0.257057,0.493162,0.721085,1.687805,...,-0.054130,1.051491,0.279553,-0.065099,1.156779,0.269777,0.000682,1.216098,0.300637,-0.007529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,-0.366221,-0.499878,0.841041,0.062922,-0.064160,-0.066458,-0.307555,0.328238,0.068514,0.130769,...,0.131416,0.097474,0.475366,0.722569,0.398540,0.918168,0.113072,0.120207,0.470438,0.001990
8986,-0.384818,-0.372495,1.951976,0.011718,-0.392493,-0.040289,-0.307314,0.427997,0.076643,0.113860,...,0.200886,0.133900,0.533365,0.681512,0.391234,0.902972,0.537189,0.357518,0.846762,0.001845
8987,-0.394956,0.176196,0.699487,-0.022420,-0.014077,-0.139570,-0.307073,0.625905,0.087201,0.110955,...,0.368262,0.215487,0.664949,0.603780,0.369073,0.863974,0.135002,0.150568,0.512915,0.002424
8988,-0.435904,-0.255453,0.698670,-0.086096,0.051910,-0.053136,-0.306833,0.430410,0.099451,0.105019,...,0.038942,0.085201,0.446056,0.514530,0.341474,0.816494,0.498140,0.366981,0.851919,0.007843


In [6]:
processed_data_test

,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,...,M5_std5,V1_E1_interaction,V1_E2_interaction,V1_E3_interaction,V2_E1_interaction,V2_E2_interaction,V2_E3_interaction,V3_E1_interaction,V3_E2_interaction,V3_E3_interaction
0,-0.974397,-0.061527,1.422118,0.244710,-1.155366,-0.439366,-1.290834,0.754881,0.131327,-1.981989,...,0.0,-0.745584,-0.809236,-0.860043,0.876723,2.213063,1.918087,1.110257,1.347795,1.296346
1,-0.093813,0.804545,-0.378890,-0.541317,-0.348686,-0.408587,-1.309741,1.373102,-0.844479,-1.225564,...,0.0,0.836342,0.928945,0.916071,0.681510,1.098135,0.999429,0.233049,0.228220,0.221847
2,1.134912,2.002584,-0.324526,-2.435383,-0.926234,0.125843,-1.296352,-1.568330,-1.896954,-1.011856,...,0.0,0.474296,0.211160,0.254168,0.524517,0.026886,0.128146,-0.004152,-0.186128,-0.157535
3,0.885654,0.072912,1.211545,-0.431209,0.303625,-0.197190,-0.018222,-0.464208,-1.166899,-0.012991,...,0.0,0.659966,0.621900,0.628019,0.299164,0.186907,0.175446,-0.915536,-0.986900,-0.984642
4,0.178209,-0.970587,-0.348617,1.869922,-0.157206,-0.529341,-0.006073,-0.462039,0.047244,0.670667,...,0.0,0.245166,0.427519,0.370595,0.112007,0.732265,0.528470,0.256904,0.405713,0.368321
5,0.422033,-0.444261,0.110109,0.566710,0.157206,0.551671,0.006073,-0.134490,-0.339243,0.819392,...,0.0,-0.401273,-0.497714,-0.527228,-0.112007,-0.026886,-0.128146,-0.815070,-0.804103,-0.818427
6,0.090388,-0.180667,0.971770,0.141687,-0.821283,0.794176,0.018215,0.134490,-0.047244,0.255490,...,0.0,-0.219264,-0.211160,-0.254168,-0.285488,-0.142061,-0.275731,0.087226,0.186128,0.157535
7,-0.090388,0.954743,0.000317,-0.141687,0.306460,-0.125843,0.030354,0.668113,0.331998,0.158623,...,0.0,0.219264,0.430657,0.365334,-0.613933,-0.491413,-0.654471,-0.768416,-0.677417,-0.708268
8,-0.820626,0.061527,-0.000317,-0.670241,0.503113,0.675126,0.042490,0.140998,0.772008,-0.039338,...,0.0,-0.643557,-0.594265,-0.665218,-0.991044,-0.926487,-1.115586,0.004152,0.225614,0.170909
9,-1.075345,-1.599362,-0.001006,3.486210,0.276590,1.065955,0.047538,-2.733189,2.026292,0.012991,...,0.0,-1.545046,-1.684851,-1.757570,-1.298782,-1.111091,-1.362998,-2.621983,-2.664135,-2.675686


In [7]:
train_split, val_split = train_test_split(
    processed_data_train, test_size=0.01, random_state=42
)

X_train = train_split.drop(columns=["market_forward_excess_returns"])
X_test = val_split.drop(columns=["market_forward_excess_returns"])
y_train = train_split['market_forward_excess_returns']
y_test = val_split['market_forward_excess_returns']

In [8]:
improved_catboost_params = {'iterations': 3000,
                            'learning_rate': 0.01,
                            'depth': 6,
                            'l2_leaf_reg': 5.0,
                            'min_child_samples': 100,
                            'colsample_bylevel': 0.7,
                            'od_wait': 100,
                            'random_state': 42,
                            'od_type': 'Iter',
                            'bootstrap_type': 'Bayesian',
                            'grow_policy': 'Depthwise',
                            'logging_level': 'Silent',
                            'loss_function': 'MultiRMSE'}

R_Forest_parm = {'n_estimators': 100,
                 'min_samples_split': 5,
                 'max_depth': 15,
                 'min_samples_leaf': 3,
                 'max_features': 'sqrt',
                 'random_state': 42}
        
Extra_parm = {'n_estimators': 100,
              'min_samples_split': 5,
              'max_depth': 12,
              'min_samples_leaf': 3,
              'max_features': 'sqrt',
              'random_state': 42}
        
XGB_R_parm = {"n_estimators": 1500,
              "learning_rate": 0.05, 
              "max_depth": 6,
              "subsample": 0.8, 
              "colsample_bytree": 0.7,
              "reg_alpha": 1.0,
              "reg_lambda": 1.0,
              "random_state": 42}

LGBM_R_parm = {"n_estimators": 1500,
               "learning_rate": 0.05,
               "num_leaves": 50,
               "max_depth": 8,
               "reg_alpha": 1.0,
               "reg_lambda": 1.0,
               "random_state": 42,
               'verbosity': -1}

DecisionTree = {'criterion': 'poisson',
                'max_depth': 6}

GB_parm = {"learning_rate": 0.1,
           "min_samples_split": 500,
           "min_samples_leaf": 50,
           "max_depth": 8,
           "max_features": 'sqrt',
           "subsample": 0.8,
           "random_state": 10}

CatBoost = CatBoostRegressor(**improved_catboost_params)
XGBoost = XGBRegressor(**XGB_R_parm)
LGBM = LGBMRegressor(**LGBM_R_parm)
RandomForest = RandomForestRegressor(**R_Forest_parm)
ExtraTrees = ExtraTreesRegressor(**Extra_parm)
GBRegressor = GradientBoostingRegressor(**GB_parm)
        
DTRegressor = DecisionTreeRegressor(**DecisionTree)


estimators = [('CatBoost', CatBoost), ('XGBoost', XGBoost), ('LGBM', LGBM), ('RandomForest', RandomForest),
              ('ExtraTrees', ExtraTrees), ('GBRegressor', GBRegressor), 
              ('ElasticNet', ElasticNetCV()), ('Lasso', Lasso()), ('RidgeCV', RidgeCV()),
              ('LassoCV', LassoCV()), ('LassoLars', LassoLars())]

model = StackingRegressor(estimators, 
                          final_estimator = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]), 
                          cv=3)
model.fit(X_train, y_train)

StackingRegressor(cv=3,
                  estimators=[('CatBoost',
                               <catboost.core.CatBoostRegressor object at 0x7d23cd7a7750>),
                              ('XGBoost',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.7, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            g...
                                                   min_samples_split=5,
                                                   random_state=42)),
                              ('GBRegressor',
                               GradientBoostingRegressor(max_depth=8,
                                                         max_features='sqrt',
                                                         min_samples_leaf=50,
                                                         min_samples_split=500,
                                                         random_state=10,
                                                         subsample=0.8)),
                              ('ElasticNet', ElasticNetCV()),
                              ('Lasso', Lasso()), ('RidgeCV', RidgeCV()),
                              ('LassoCV', LassoCV()),
                              ('LassoLars', LassoLars())],
                  final_estimator=RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]))

In [9]:
def predict(test: pl.DataFrame) -> float:
    test = test.to_pandas()
    test, metadata_train = preprocessing(test, "test")
    raw_pred = model.predict(test)[0]
    return raw_pred

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

Processing 1 rows with 86 features
Missing values before processing: 0
Added feature engineering - new shape: (1, 105)
Final shape: (1, 105)
Missing values after processing: 10
Data types: {dtype('float64'): 105}
Processing 1 rows with 86 features
Missing values before processing: 0
Added feature engineering - new shape: (1, 105)
Final shape: (1, 105)
Missing values after processing: 10
Data types: {dtype('float64'): 105}
Processing 1 rows with 86 features
Missing values before processing: 0
Added feature engineering - new shape: (1, 105)
Final shape: (1, 105)
Missing values after processing: 10
Data types: {dtype('float64'): 105}
Processing 1 rows with 86 features
Missing values before processing: 0
Added feature engineering - new shape: (1, 105)
Final shape: (1, 105)
Missing values after processing: 10
Data types: {dtype('float64'): 105}
Processing 1 rows with 86 features
Missing values before processing: 0
Added feature engineering - new shape: (1, 105)
Final shape: (1, 105)
Missing